# Portable WHOOSH Parameter Tuning Demo

This notebook is self-contained inside the `final_demo` folder.

It does not use hardcoded machine paths. It loads local scripts and relative paths from `demo_config.yaml`.

Use it to explain WHOOSH parameter behavior to business and technical teams.

## 1. Setup

Run this cell first. It finds the portable demo folder and loads `demo_config.yaml`.

In [ ]:
from __future__ import annotations

import json
import sys
import tempfile
from itertools import product
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display


def find_demo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents, current / "final_demo"]:
        if (candidate / "demo_config.yaml").exists() and (candidate / "whoosh_demo_engine.py").exists():
            return candidate
        nested = candidate / "final_demo"
        if (nested / "demo_config.yaml").exists() and (nested / "whoosh_demo_engine.py").exists():
            return nested
    raise FileNotFoundError("Could not find demo_config.yaml. Open or run this notebook from the final_demo folder.")


DEMO_ROOT = find_demo_root()
sys.path.insert(0, str(DEMO_ROOT))

from whoosh_demo_engine import DetectionSettings, make_demo_ocr_json, run_keyword_detection, write_json_file
from metrics_demo import compare_output_dir_vs_gt

config = yaml.safe_load((DEMO_ROOT / "demo_config.yaml").read_text(encoding="utf-8"))

print("Demo root:", DEMO_ROOT)
display(pd.DataFrame(config["parameters"].items(), columns=["parameter", "configured_values"]))

## 2. Helper Functions

These helpers create tiny OCR examples, run WHOOSH, and show which pages matched.

In [ ]:
def run_demo(
    pages: dict[int, str],
    keywords: dict[str, list[str]],
    *,
    slop: int = 1,
    edit_distance: int = 0,
    min_fuzzy_term_length: int = 99,
    keep_stopwords: bool = True,
    stem_words: bool = True,
    prefixlength: int = 0,
) -> pd.DataFrame:
    with tempfile.TemporaryDirectory() as temp_dir:
        root = Path(temp_dir)
        ocr_path = root / "ocr.json"
        keywords_path = root / "keywords.json"
        write_json_file(ocr_path, make_demo_ocr_json(pages))
        write_json_file(keywords_path, keywords)

        settings = DetectionSettings(
            slop=slop,
            edit_distance=edit_distance,
            min_fuzzy_term_length=min_fuzzy_term_length,
            keep_stopwords=keep_stopwords,
            stem_words=stem_words,
            prefixlength=prefixlength,
            matched_only=False,
            include_file_path=False,
        )
        output = run_keyword_detection(ocr_path, keywords_path, settings)

    output_by_page = {record["page_number"]: record for record in output}
    rows = []
    for page_number, text in pages.items():
        matches = output_by_page.get(page_number, {"matches": []})["matches"]
        rows.append(
            {
                "page_number": page_number,
                "ocr_text": text,
                "detected_groups": ", ".join(match["group"] for match in matches),
                "detected_variants": ", ".join(match["variant"] for match in matches),
                "match_count": len(matches),
            }
        )
    return pd.DataFrame(rows)


def matched_pages(result_df: pd.DataFrame) -> list[int]:
    return result_df.loc[result_df["match_count"] > 0, "page_number"].tolist()


def summarize_runs(rows: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(rows)

## 3. Single-Word Keyword

`slop` does not matter for a single-word keyword because there is only one word.

Example keyword: `Emergency`.

In [ ]:
single_word_pages = {
    1: "Contact Type: EMERGENCY",
    2: "Routine follow up visit",
}
single_word_keywords = {"emergent": ["Emergency"]}

rows = []
for slop in [1, 5]:
    result = run_demo(single_word_pages, single_word_keywords, slop=slop, edit_distance=0, min_fuzzy_term_length=99)
    rows.append({"slop": slop, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))

## 4. Multi-Word Keyword: `slop`

`slop` controls how close words must be for a multi-word keyword.

- `slop=1`: words must be adjacent.
- `slop=2`: allows a small gap.
- `slop=5`: allows a wider gap.

The code uses unordered matching, so OCR word order can vary as long as words are close enough.

In [ ]:
slop_pages = {
    1: "signed by physician",
    2: "signed electronically by physician",
    3: "signed electronically after review by physician",
}
slop_keywords = {"signature": ["signed by"]}

rows = []
for slop in [1, 2, 5]:
    result = run_demo(
        slop_pages,
        slop_keywords,
        slop=slop,
        edit_distance=0,
        min_fuzzy_term_length=99,
        keep_stopwords=True,
    )
    rows.append({"slop": slop, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))
display(result)

## 5. `edit_distance`: OCR Typo Tolerance

`edit_distance` means how many character differences are allowed inside a word.

- `0`: exact word only.
- `1`: one typo is allowed.
- `2`: up to two typos are allowed.

In [ ]:
edit_distance_pages = {
    1: "Physician note signed",
    2: "Physican note signed",
    3: "Phsician note signed",
    4: "Pysican note signed",
}
edit_distance_keywords = {"provider": ["Physician"]}

rows = []
for edit_distance in [0, 1, 2]:
    result = run_demo(
        edit_distance_pages,
        edit_distance_keywords,
        slop=1,
        edit_distance=edit_distance,
        min_fuzzy_term_length=1,
        keep_stopwords=True,
    )
    rows.append({"edit_distance": edit_distance, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))
display(result)

## 6. `min_fuzzy_term_length`: Avoid Risky Short-Word Fuzzy Matches

If a word is shorter than `min_fuzzy_term_length`, it must match exactly.

This protects short terms like `STAT`, which are close to `START` and `STATE`.

In [ ]:
min_fuzzy_pages = {
    1: "STAT order",
    2: "START order",
    3: "STATE order",
}
min_fuzzy_keywords = {"urgent": ["STAT"]}

rows = []
for min_fuzzy_term_length in [1, 5]:
    result = run_demo(
        min_fuzzy_pages,
        min_fuzzy_keywords,
        slop=1,
        edit_distance=1,
        min_fuzzy_term_length=min_fuzzy_term_length,
        keep_stopwords=True,
    )
    rows.append({"min_fuzzy_term_length": min_fuzzy_term_length, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))
display(result)

## 7. `keep_stopwords`: Keep or Remove Common Words

Stop words are common words like `by`, `the`, `and`, `of`, `to`.

For phrases such as `signed by`, the word `by` matters. Removing it can make the keyword behave like only `signed`.

In [ ]:
stopword_pages = {
    1: "signed by physician",
    2: "signed yesterday by physician",
    3: "signed yesterday",
}
stopword_keywords = {"signature": ["signed by"]}

rows = []
for keep_stopwords in [False, True]:
    result = run_demo(
        stopword_pages,
        stopword_keywords,
        slop=1,
        edit_distance=0,
        min_fuzzy_term_length=99,
        keep_stopwords=keep_stopwords,
    )
    rows.append({"keep_stopwords": keep_stopwords, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))
display(result)

## 8. `stem_words`: Match Related Word Forms

Production tuning keeps `stem_words=true`.

This example turns it off only to explain the effect.

In [ ]:
stem_pages = {
    1: "review complete",
    2: "reviewed by nurse",
    3: "reviewing case",
}
stem_keywords = {"review_status": ["review"]}

rows = []
for stem_words in [False, True]:
    result = run_demo(
        stem_pages,
        stem_keywords,
        slop=1,
        edit_distance=0,
        min_fuzzy_term_length=1,
        keep_stopwords=True,
        stem_words=stem_words,
    )
    rows.append({"stem_words": stem_words, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))
display(result)

## 9. `prefixlength`: Require the Beginning of the Word

The current production-like config uses `prefixlength=0`.

`prefixlength=1` means the first character must match exactly before fuzzy matching is allowed.

In [ ]:
prefix_pages = {
    1: "Physician note",
    2: "Xhysician note",
    3: "Physican note",
}
prefix_keywords = {"provider": ["Physician"]}

rows = []
for prefixlength in [0, 1, 2]:
    result = run_demo(
        prefix_pages,
        prefix_keywords,
        slop=1,
        edit_distance=1,
        min_fuzzy_term_length=1,
        keep_stopwords=True,
        prefixlength=prefixlength,
    )
    rows.append({"prefixlength": prefixlength, "matched_pages": matched_pages(result)})

display(summarize_runs(rows))
display(result)

## 10. Page-Level vs Keyword-Level Metrics

This uses files under `sample_data/` and paths from `demo_config.yaml`.

Page level asks whether a page has at least one keyword.

Keyword level asks whether the correct keyword label was found on the correct page.

In [ ]:
sample_gt_path = DEMO_ROOT / config["paths"]["sample_gt"]
sample_method_path = DEMO_ROOT / config["paths"]["sample_method_output"]

metrics = compare_output_dir_vs_gt(
    method_output_dir=sample_method_path.parent,
    gt_output_dir=sample_gt_path.parent,
    method_glob="*_keyword_output.json",
    preferred_match_field=config["report"]["method_key_field"],
).to_dict()

display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

Expected interpretation for the sample files:

- GT positive pages are page 3 and page 5.
- Method detects page 3 only.
- Page-level: `TP=1`, `FP=0`, `FN=1`, and remaining pages are `TN`.
- Keyword-level: `TP=1` for `emergent`, `FN=1` for missed `immediate`.

## 11. Run a Tiny Parameter Grid

This is not the full production run. It is a small live demo showing that different parameter combinations change detections.

In [ ]:
tiny_pages = {
    1: "signed by physician",
    2: "signed electronically by physician",
    3: "Physican note signed",
    4: "START order",
}
tiny_keywords = {
    "signature": ["signed by"],
    "provider": ["Physician"],
    "urgent": ["STAT"],
}

grid_rows = []
for slop, edit_distance, min_fuzzy_term_length, keep_stopwords in product(
    config["parameters"]["slop"],
    config["parameters"]["edit_distance"],
    config["parameters"]["min_fuzzy_term_length"],
    config["parameters"]["keep_stopwords"],
):
    result = run_demo(
        tiny_pages,
        tiny_keywords,
        slop=slop,
        edit_distance=edit_distance,
        min_fuzzy_term_length=min_fuzzy_term_length,
        keep_stopwords=keep_stopwords,
        stem_words=config["parameters"]["stem_words"],
        prefixlength=config["parameters"]["prefixlength"],
    )
    detected_groups = sorted(
        {
            group.strip()
            for value in result["detected_groups"]
            for group in value.split(",")
            if group.strip()
        }
    )
    grid_rows.append(
        {
            "slop": slop,
            "edit_distance": edit_distance,
            "min_fuzzy_term_length": min_fuzzy_term_length,
            "keep_stopwords": keep_stopwords,
            "matched_pages": matched_pages(result),
            "matched_page_count": len(matched_pages(result)),
            "detected_groups": ", ".join(detected_groups),
        }
    )

grid_df = pd.DataFrame(grid_rows)
print("Grid rows:", len(grid_df))
display(grid_df.head(20))
display(
    grid_df.groupby(["matched_page_count", "detected_groups"])
    .size()
    .reset_index(name="combination_count")
    .sort_values(["matched_page_count", "combination_count"], ascending=[False, False])
)

## Business Takeaway

Parameter tuning is testing strictness.

Stricter settings usually reduce false positives but can miss OCR-damaged text.

Broader settings usually recover more OCR-damaged text but can introduce false positives.

The best production setting is the one with the best balance against ground truth.